In [1]:
import pandas as pd

jobs = pd.read_parquet("data/prepped/jobs.parquet")

print(jobs.shape)
display(jobs.head())
print(jobs.columns.tolist())

(74849, 65)


,id_job,gpu_count,gpu_hours,energy_wh,exec_sec,sm_util_avg,sm_util_max,mem_util_avg,max_gpu_mem_used,mem_used_frac,...,mem_req_mb,mem_req_is_per_cpu,mem_req_total_mb,is_array_task,wait_sec,walltime_sec,primary_node,n_nodes_listed,gpu_hours_alloc,gpu_hours_ratio
0,1055011,1,0.000153,0.003817,0.55,0.0,0.0,0.0,0.000000e+00,0.000000,...,9600,True,9600,False,1.0,0.0,r368967-n172107,1,0.000000,NaN
1,1363534,1,3.584472,778.816202,12904.10,96.0,99.0,64.0,4.480565e+09,0.130402,...,8500,True,34000,True,2.0,12904.0,r368967-n750018,1,3.584444,1.000008
2,1808972,1,0.276128,24.616018,994.06,27.0,93.0,13.0,4.780458e+09,0.139130,...,8500,True,170000,False,0.0,971.0,r810901-n948219,1,0.269722,1.023749
3,2688926,2,0.027644,0.718045,49.76,0.0,0.0,0.0,0.000000e+00,0.000000,...,8500,True,340000,False,6.0,49.0,r5179276-n200569,1,0.027222,1.015510
4,3084235,1,35.084167,1276.702300,126303.00,1.0,100.0,0.0,6.286213e+09,0.182953,...,8500,True,85000,True,4078.0,126303.0,r6959555-n200569,1,35.084167,1.000000


['id_job', 'gpu_count', 'gpu_hours', 'energy_wh', 'exec_sec', 'sm_util_avg', 'sm_util_max', 'mem_util_avg', 'max_gpu_mem_used', 'mem_used_frac', 'watts_avg', 'watts_max', 'pcie_rx_avg', 'pcie_tx_avg', 'n_nodes_dcgm', 'id_array_job', 'id_array_task', 'id_user', 'kill_requid', 'nodes_alloc', 'nodelist', 'cpus_req', 'derived_ec', 'exit_code', 'gres_req', 'gres_alloc', 'gres_used', 'array_max_tasks', 'array_task_pending', 'constraints', 'flags', 'mem_req', 'partition', 'priority', 'state', 'timelimit', 'time_submit', 'time_eligible', 'time_start', 'time_end', 'time_suspended', 'track_steps', 'tres_alloc', 'tres_req', 'job_type', 'attempts', 'nodefail_attempts', 'hit_node_failure', 'nodefail_nodes', 'nodefail_exact', 'nodefail_last_end', 'nodefail_wall_sec', 'state_name', 'is_terminal', 'is_success', 'mem_req_mb', 'mem_req_is_per_cpu', 'mem_req_total_mb', 'is_array_task', 'wait_sec', 'walltime_sec', 'primary_node', 'n_nodes_listed', 'gpu_hours_alloc', 'gpu_hours_ratio']


In [2]:
total_gpu_hours = jobs["gpu_hours"].sum()

print("Total GPU-hours:", total_gpu_hours)

Total GPU-hours: 594003.8401444445


In [3]:
baseline = (
    jobs.groupby("state_name")
        .agg(
            jobs=("id_job", "count"),
            gpu_hours=("gpu_hours", "sum")
        )
        .sort_values("gpu_hours", ascending=False)
)

baseline["share_pct"] = baseline["gpu_hours"] / total_gpu_hours * 100
baseline["usd"] = baseline["gpu_hours"] * 2.5

baseline

,jobs,gpu_hours,share_pct,usd
state_name,,,,
COMPLETED,45334,229040.571769,38.558770,572601.429424
CANCELLED,9290,203929.576064,34.331357,509823.940160
TIMEOUT,1544,107951.521208,18.173539,269878.803021
FAILED,18587,50033.157061,8.423036,125082.892653
NODE_FAIL,10,2027.890883,0.341394,5069.727208
UNDECODED_11,83,1021.114400,0.171904,2552.786000
UNDECODED_1024,1,0.008758,0.000001,0.021896


In [4]:
import json

with open("data/synthetic/findings.json") as f:
    findings = json.load(f)

type(findings)

list

In [5]:
len(findings)

11979

In [6]:
gpu_not_needed = [
    f for f in findings
    if f.get("detectorId") == "rules::gpu-not-needed"
]

print("Number of findings:", len(gpu_not_needed))

Number of findings: 463


In [7]:
gpu_not_needed_hours = sum(
    f.get("metadata", {}).get("impact_gpu_hours", 0) or 0
    for f in gpu_not_needed
)

gpu_not_needed_usd = gpu_not_needed_hours * 2.5

print("GPU-hours:", gpu_not_needed_hours)
print("USD:", gpu_not_needed_usd)
print("% of total capacity:", gpu_not_needed_hours / total_gpu_hours * 100)

GPU-hours: 11986.24
USD: 29965.6
% of total capacity: 2.017872476562659


In [8]:
print({
    f.get("metadata", {}).get("impact_kind")
    for f in gpu_not_needed
})

print({
    f.get("metadata", {}).get("impact_scope")
    for f in gpu_not_needed
})

{'unused_capacity'}
{'job'}


In [9]:
idle_interactive = [
    f for f in findings
    if f.get("detectorId") == "rules::idle-interactive-session"
]

print("Number of findings:", len(idle_interactive))

Number of findings: 906


In [10]:
idle_interactive_hours = sum(
    f.get("metadata", {}).get("impact_gpu_hours", 0) or 0
    for f in idle_interactive
)

idle_interactive_usd = idle_interactive_hours * 2.5

print("GPU-hours:", idle_interactive_hours)
print("USD:", idle_interactive_usd)
print("% of total capacity:", idle_interactive_hours / total_gpu_hours * 100)

GPU-hours: 36663.16
USD: 91657.90000000001
% of total capacity: 6.172209255597504


In [11]:
print({
    f.get("metadata", {}).get("impact_kind")
    for f in idle_interactive
})

print({
    f.get("metadata", {}).get("impact_scope")
    for f in idle_interactive
})

{'unused_capacity'}
{'job'}


In [12]:
gpu_not_needed_ids = {
    f.get("metadata", {}).get("job_id")
    for f in gpu_not_needed
}

idle_interactive_ids = {
    f.get("metadata", {}).get("job_id")
    for f in idle_interactive
}

overlap_ids = gpu_not_needed_ids & idle_interactive_ids

print("GPU not needed unique jobs:", len(gpu_not_needed_ids))
print("Idle interactive unique jobs:", len(idle_interactive_ids))
print("Overlap jobs:", len(overlap_ids))

GPU not needed unique jobs: 463
Idle interactive unique jobs: 906
Overlap jobs: 109


In [13]:
import pandas as pd

selected_findings = gpu_not_needed + idle_interactive

rows = []

for f in selected_findings:
    meta = f.get("metadata", {})
    rows.append({
        "job_id": meta.get("job_id"),
        "rule": f.get("detectorId"),
        "impact_gpu_hours": meta.get("impact_gpu_hours", 0) or 0
    })

impact_df = pd.DataFrame(rows)

impact_df.head()

,job_id,rule,impact_gpu_hours
0,24414683,rules::gpu-not-needed,1.14
1,178571300,rules::gpu-not-needed,3.05
2,323631158,rules::gpu-not-needed,18.38
3,521294447,rules::gpu-not-needed,7.58
4,908435527,rules::gpu-not-needed,12.83


In [14]:
dedup = (
    impact_df.groupby("job_id", as_index=False)
             ["impact_gpu_hours"]
             .max()
)

combined_hours = dedup["impact_gpu_hours"].sum()

print("Unique jobs:", dedup["job_id"].nunique())
print("Deduplicated GPU-hours:", combined_hours)
print("USD:", combined_hours * 2.5)
print("% total capacity:", combined_hours / total_gpu_hours * 100)

Unique jobs: 1260
Deduplicated GPU-hours: 45114.35
USD: 112785.875
% total capacity: 7.594959316934633


In [16]:
overlap_df = impact_df[
    impact_df["job_id"].isin(overlap_ids)
]

overlap_pivot = overlap_df.pivot(
    index="job_id",
    columns="rule",
    values="impact_gpu_hours"
)

display(overlap_pivot.head())

print(
    "Same impact:",
    (
        overlap_pivot["rules::gpu-not-needed"]
        == overlap_pivot["rules::idle-interactive-session"]
    ).all()
)

rule,rules::gpu-not-needed,rules::idle-interactive-session
job_id,,
323631158,18.38,18.38
1101448790,57.84,57.84
2426207699,21.22,21.22
2748402495,24.06,24.06
3333444007,9.93,9.93


Same impact: True


In [17]:
gpu_never_computed = [
    f for f in findings
    if f.get("detectorId") == "rules::gpu-never-computed"
]

print("Number of findings:", len(gpu_never_computed))

gpu_never_computed_hours = sum(
    f.get("metadata", {}).get("impact_gpu_hours", 0) or 0
    for f in gpu_never_computed
)

print("GPU-hours:", gpu_never_computed_hours)
print("USD:", gpu_never_computed_hours * 2.5)
print("% total capacity:", gpu_never_computed_hours / total_gpu_hours * 100)

Number of findings: 1459
GPU-hours: 81887.01
USD: 204717.525
% total capacity: 13.785602796791254


In [18]:
gpu_never_ids = {
    f.get("metadata", {}).get("job_id")
    for f in gpu_never_computed
}

never_jobs = jobs[
    jobs["id_job"].isin(gpu_never_ids)
]

never_breakdown = (
    never_jobs.groupby("state_name")
    .agg(
        jobs=("id_job", "count"),
        gpu_hours=("gpu_hours", "sum")
    )
    .sort_values("gpu_hours", ascending=False)
)

never_breakdown["share_pct"] = (
    never_breakdown["gpu_hours"]
    / never_breakdown["gpu_hours"].sum()
    * 100
)

never_breakdown["usd"] = never_breakdown["gpu_hours"] * 2.5

never_breakdown

,jobs,gpu_hours,share_pct,usd
state_name,,,,
CANCELLED,312,34824.537867,42.527592,87061.344667
TIMEOUT,820,30640.657133,37.418253,76601.642833
FAILED,311,15369.134583,18.768728,38422.836458
UNDECODED_11,14,805.569167,0.983758,2013.922917
NODE_FAIL,2,247.027528,0.301669,617.568819


In [19]:
gpu_never_ids = {
    f.get("metadata", {}).get("job_id")
    for f in gpu_never_computed
}

never_idle_overlap = gpu_never_ids & idle_interactive_ids

print("GPU-never-computed unique jobs:", len(gpu_never_ids))
print("Overlap with idle interactive:", len(never_idle_overlap))

GPU-never-computed unique jobs: 1459
Overlap with idle interactive: 528


In [20]:
never_idle_ids = gpu_never_ids & idle_interactive_ids

never_idle_jobs = jobs[
    jobs["id_job"].isin(never_idle_ids)
]

print("Overlap jobs:", len(never_idle_ids))
print("Overlap GPU-hours:", never_idle_jobs["gpu_hours"].sum())
print("Overlap USD:", never_idle_jobs["gpu_hours"].sum() * 2.5)

Overlap jobs: 528
Overlap GPU-hours: 27384.753194444445
Overlap USD: 68461.88298611112


In [21]:
slow_cancel = [
    f for f in findings
    if f.get("detectorId") == "rules::slow-cancel-of-idle-job"
]

slow_cancel_hours = sum(
    f.get("metadata", {}).get("impact_gpu_hours", 0) or 0
    for f in slow_cancel
)

slow_cancel_ids = {
    f.get("metadata", {}).get("job_id")
    for f in slow_cancel
}

print("Findings:", len(slow_cancel))
print("Unique jobs:", len(slow_cancel_ids))
print("GPU-hours:", slow_cancel_hours)
print("USD:", slow_cancel_hours * 2.5)
print("% total capacity:", slow_cancel_hours / total_gpu_hours * 100)

print(
    "Overlap with gpu-never-computed:",
    len(slow_cancel_ids & gpu_never_ids)
)

Findings: 949
Unique jobs: 949
GPU-hours: 71052.53
USD: 177631.325
% total capacity: 11.96162805660012
Overlap with gpu-never-computed: 311


In [22]:
print({
    f.get("metadata", {}).get("impact_kind")
    for f in slow_cancel
})

print({
    f.get("metadata", {}).get("impact_scope")
    for f in slow_cancel
})

{'unused_capacity'}
{'job'}


In [23]:
print(
    "Overlap with idle interactive:",
    len(slow_cancel_ids & idle_interactive_ids)
)

print(
    "Overlap with gpu not needed:",
    len(slow_cancel_ids & gpu_not_needed_ids)
)

Overlap with idle interactive: 58
Overlap with gpu not needed: 0


In [24]:
unused_findings = (
    gpu_not_needed
    + idle_interactive
    + slow_cancel
)

rows = []

for f in unused_findings:
    meta = f.get("metadata", {})
    rows.append({
        "job_id": meta.get("job_id"),
        "rule": f.get("detectorId"),
        "impact_gpu_hours": meta.get("impact_gpu_hours", 0) or 0
    })

unused_df = pd.DataFrame(rows)

unused_dedup = (
    unused_df
    .groupby("job_id", as_index=False)["impact_gpu_hours"]
    .max()
)

unused_hours = unused_dedup["impact_gpu_hours"].sum()

print("Unique jobs:", unused_dedup["job_id"].nunique())
print("Deduplicated unused GPU-hours:", unused_hours)
print("USD:", unused_hours * 2.5)
print("% total capacity:", unused_hours / total_gpu_hours * 100)

Unique jobs: 2151
Deduplicated unused GPU-hours: 111572.37000000001
USD: 278930.92500000005
% total capacity: 18.783105842020962


In [25]:
unused_job_ids = set(unused_dedup["job_id"])

unused_real_jobs = jobs[
    jobs["id_job"].isin(unused_job_ids)
]

real_gpu_hours = unused_real_jobs["gpu_hours"].sum()

print("Finding-based dedup GPU-hours:", unused_hours)
print("Actual job GPU-hours:", real_gpu_hours)
print("Difference:", unused_hours - real_gpu_hours)

Finding-based dedup GPU-hours: 111572.37000000001
Actual job GPU-hours: 111572.10251944445
Difference: 0.2674805555579951


In [26]:
gpu_low_util = [
    f for f in findings
    if f.get("detectorId") == "rules::gpu-low-utilization"
]

gpu_low_util_hours = sum(
    f.get("metadata", {}).get("impact_gpu_hours", 0) or 0
    for f in gpu_low_util
)

gpu_low_util_ids = {
    f.get("metadata", {}).get("job_id")
    for f in gpu_low_util
}

print("Findings:", len(gpu_low_util))
print("Unique jobs:", len(gpu_low_util_ids))
print("GPU-hours:", gpu_low_util_hours)
print("USD:", gpu_low_util_hours * 2.5)
print("% total capacity:", gpu_low_util_hours / total_gpu_hours * 100)

Findings: 95
Unique jobs: 95
GPU-hours: 61062.89
USD: 152657.225
% total capacity: 10.27988135314938


In [27]:
low_never_overlap = gpu_low_util_ids & gpu_never_ids

print("Overlap with gpu-never-computed:", len(low_never_overlap))

low_never_jobs = jobs[
    jobs["id_job"].isin(low_never_overlap)
]

print("Overlap GPU-hours:", low_never_jobs["gpu_hours"].sum())
print("Overlap USD:", low_never_jobs["gpu_hours"].sum() * 2.5)

Overlap with gpu-never-computed: 48
Overlap GPU-hours: 26440.072777777776
Overlap USD: 66100.18194444444


In [28]:
low_util_jobs = jobs[
    jobs["id_job"].isin(gpu_low_util_ids)
]

low_util_breakdown = (
    low_util_jobs.groupby("state_name")
    .agg(
        jobs=("id_job", "count"),
        gpu_hours=("gpu_hours", "sum")
    )
    .sort_values("gpu_hours", ascending=False)
)

low_util_breakdown["share_pct"] = (
    low_util_breakdown["gpu_hours"]
    / low_util_breakdown["gpu_hours"].sum()
    * 100
)

low_util_breakdown["usd"] = low_util_breakdown["gpu_hours"] * 2.5

low_util_breakdown

,jobs,gpu_hours,share_pct,usd
state_name,,,,
CANCELLED,58,41730.767000,68.340618,104326.917500
COMPLETED,19,8776.257278,14.372486,21940.643194
TIMEOUT,7,5376.215000,8.804388,13440.537500
FAILED,9,3987.075000,6.529455,9967.687500
NODE_FAIL,2,1192.591111,1.953053,2981.477778


In [29]:
low_unused_overlap = gpu_low_util_ids & unused_job_ids

low_unused_jobs = jobs[
    jobs["id_job"].isin(low_unused_overlap)
]

print("Overlap with unused-capacity bucket:", len(low_unused_overlap))
print("Overlap GPU-hours:", low_unused_jobs["gpu_hours"].sum())
print("Overlap USD:", low_unused_jobs["gpu_hours"].sum() * 2.5)

Overlap with unused-capacity bucket: 64
Overlap GPU-hours: 42167.824222222225
Overlap USD: 105419.56055555557


In [30]:
completed_low_util = low_util_jobs[
    low_util_jobs["state_name"] == "COMPLETED"
]

print("Completed low-util jobs:", len(completed_low_util))
print("GPU-hours:", completed_low_util["gpu_hours"].sum())
print("USD:", completed_low_util["gpu_hours"].sum() * 2.5)
print(
    "% total capacity:",
    completed_low_util["gpu_hours"].sum() / total_gpu_hours * 100
)

Completed low-util jobs: 19
GPU-hours: 8776.257277777777
USD: 21940.64319444444
% total capacity: 1.4774748384865064


In [31]:
completed_low_ids = set(completed_low_util["id_job"])

completed_low_unused_overlap = completed_low_ids & unused_job_ids

print("Completed low-util jobs:", len(completed_low_ids))
print(
    "Already in unused bucket:",
    len(completed_low_unused_overlap)
)

clean_completed_low_ids = completed_low_ids - unused_job_ids

clean_completed_low = jobs[
    jobs["id_job"].isin(clean_completed_low_ids)
]

print("New right-sizing jobs:", len(clean_completed_low))
print("GPU-hours:", clean_completed_low["gpu_hours"].sum())
print("USD:", clean_completed_low["gpu_hours"].sum() * 2.5)
print(
    "% total capacity:",
    clean_completed_low["gpu_hours"].sum()
    / total_gpu_hours * 100
)

Completed low-util jobs: 19
Already in unused bucket: 5
New right-sizing jobs: 14
GPU-hours: 6806.382277777776
USD: 17015.95569444444
% total capacity: 1.1458481945373724


In [32]:
gpu_mem_oversized = [
    f for f in findings
    if f.get("detectorId") == "rules::gpu-memory-oversized"
]

gpu_mem_oversized_ids = {
    f.get("metadata", {}).get("job_id")
    for f in gpu_mem_oversized
}

gpu_mem_oversized_hours = sum(
    f.get("metadata", {}).get("impact_gpu_hours", 0) or 0
    for f in gpu_mem_oversized
)

print("Findings:", len(gpu_mem_oversized))
print("Unique jobs:", len(gpu_mem_oversized_ids))
print("GPU-hours:", gpu_mem_oversized_hours)
print("USD:", gpu_mem_oversized_hours * 2.5)
print(
    "% total capacity:",
    gpu_mem_oversized_hours / total_gpu_hours * 100
)

Findings: 108
Unique jobs: 108
GPU-hours: 107008.58
USD: 267521.45
% total capacity: 18.01479599424452


In [33]:
# 1. 确认 impact kind
print({
    f.get("metadata", {}).get("impact_kind")
    for f in gpu_mem_oversized
})

# 2. 和 low-utilization overlap
mem_low_overlap = gpu_mem_oversized_ids & gpu_low_util_ids

mem_low_jobs = jobs[
    jobs["id_job"].isin(mem_low_overlap)
]

print("Overlap with low-util:", len(mem_low_overlap))
print("Overlap GPU-hours:", mem_low_jobs["gpu_hours"].sum())
print("Overlap USD:", mem_low_jobs["gpu_hours"].sum() * 2.5)

# 3. 和 Bucket A unused capacity overlap
mem_unused_overlap = gpu_mem_oversized_ids & unused_job_ids

mem_unused_jobs = jobs[
    jobs["id_job"].isin(mem_unused_overlap)
]

print("Overlap with unused bucket:", len(mem_unused_overlap))
print("Overlap GPU-hours:", mem_unused_jobs["gpu_hours"].sum())
print("Overlap USD:", mem_unused_jobs["gpu_hours"].sum() * 2.5)

{'unused_capacity'}
Overlap with low-util: 43
Overlap GPU-hours: 26658.89688888889
Overlap USD: 66647.24222222222
Overlap with unused bucket: 30
Overlap GPU-hours: 18566.832444444444
Overlap USD: 46417.08111111111


In [34]:
mem_jobs = jobs[
    jobs["id_job"].isin(gpu_mem_oversized_ids)
]

mem_breakdown = (
    mem_jobs.groupby("state_name")
    .agg(
        jobs=("id_job", "count"),
        gpu_hours=("gpu_hours", "sum")
    )
    .sort_values("gpu_hours", ascending=False)
)

mem_breakdown["share_pct"] = (
    mem_breakdown["gpu_hours"]
    / mem_breakdown["gpu_hours"].sum()
    * 100
)

mem_breakdown["usd"] = mem_breakdown["gpu_hours"] * 2.5

mem_breakdown

,jobs,gpu_hours,share_pct,usd
state_name,,,,
CANCELLED,67,62420.920667,56.379554,156052.301667
TIMEOUT,11,33863.544222,30.586084,84658.860556
FAILED,11,7140.757500,6.449644,17851.893750
COMPLETED,17,6654.731667,6.010658,16636.829167
NODE_FAIL,2,635.572778,0.574059,1588.931944


In [35]:
gpu_mem_oversized[0]

{'id': '9907f2ee-2cbf-5618-89ab-c79e39325a60',
 'integrationId': 'dc59fe21-aef3-56fb-a6e5-7de22e87047d',
 'enterpriseId': '17372250-e094-56f6-9f3c-3907f9017fd1',
 'detectorId': 'rules::gpu-memory-oversized',
 'shortDescription': 'Job used 7% of available GPU memory',
 'longDescription': 'Job 805875933 peaked at 6.5% of the 32 GB available per V100 across 313.4 GPU-hours.',
 'impactDescription': '292.9 GPU-hours of memory capacity unused.',
 'resourceIds': ['bd7fba17-3748-5ada-ab4a-6494d35c6995'],
 'rootCauses': [],
 'status': 'ACTION_REQUIRED',
 'severity': 'LOW',
 'confidence': 'MEDIUM',
 'category': 'PERFORMANCE',
 'priority': 'P4',
 'detectionTime': '2026-06-22T09:18:20Z',
 'isActive': True,
 'metadata': {'mem_used_frac': 0.0654,
  'gpu_hours': 313.44,
  'synthetic': False,
  'impact_gpu_hours': 292.94,
  'impact_kind': 'unused_capacity',
  'impact_scope': 'job',
  'job_id': 805875933}}

In [36]:
print(gpu_mem_oversized[0]["detectorId"])
print(gpu_mem_oversized[0]["metadata"])

rules::gpu-memory-oversized
{'mem_used_frac': 0.0654, 'gpu_hours': 313.44, 'synthetic': False, 'impact_gpu_hours': 292.94, 'impact_kind': 'unused_capacity', 'impact_scope': 'job', 'job_id': 805875933}


In [37]:
completed_mem = mem_jobs[
    mem_jobs["state_name"] == "COMPLETED"
]

completed_mem_ids = set(completed_mem["id_job"])

print("Completed memory-oversized jobs:", len(completed_mem_ids))
print("Full job GPU-hours:", completed_mem["gpu_hours"].sum())

Completed memory-oversized jobs: 17
Full job GPU-hours: 6654.731666666666


In [38]:
completed_mem_findings = [
    f for f in gpu_mem_oversized
    if f.get("metadata", {}).get("job_id") in completed_mem_ids
]

completed_mem_impact_hours = sum(
    f.get("metadata", {}).get("impact_gpu_hours", 0) or 0
    for f in completed_mem_findings
)

print("Recoverable impact GPU-hours:", completed_mem_impact_hours)
print("USD:", completed_mem_impact_hours * 2.5)
print("% total capacity:", completed_mem_impact_hours / total_gpu_hours * 100)

Recoverable impact GPU-hours: 6269.21
USD: 15673.025
% total capacity: 1.0554157357763059


In [39]:
clean_completed_mem_ids = completed_mem_ids - unused_job_ids

print("Completed memory-oversized jobs:", len(completed_mem_ids))
print("Already in Bucket A:", len(completed_mem_ids & unused_job_ids))
print("New jobs:", len(clean_completed_mem_ids))

Completed memory-oversized jobs: 17
Already in Bucket A: 3
New jobs: 14


In [40]:
clean_completed_mem_findings = [
    f for f in gpu_mem_oversized
    if f.get("metadata", {}).get("job_id") in clean_completed_mem_ids
]

clean_mem_impact_hours = sum(
    f.get("metadata", {}).get("impact_gpu_hours", 0) or 0
    for f in clean_completed_mem_findings
)

print("New memory-rightsizing jobs:", len(clean_completed_mem_ids))
print("New impact GPU-hours:", clean_mem_impact_hours)
print("USD:", clean_mem_impact_hours * 2.5)
print("% total capacity:", clean_mem_impact_hours / total_gpu_hours * 100)

New memory-rightsizing jobs: 14
New impact GPU-hours: 5183.27
USD: 12958.175000000001
% total capacity: 0.8725987358498524


In [41]:
clean_completed_low_ids = set(clean_completed_low["id_job"])

low_mem_overlap = (
    clean_completed_low_ids
    & clean_completed_mem_ids
)

print("Overlap jobs:", len(low_mem_overlap))

overlap_jobs = jobs[
    jobs["id_job"].isin(low_mem_overlap)
]

print("Overlap full GPU-hours:",
      overlap_jobs["gpu_hours"].sum())

Overlap jobs: 8
Overlap full GPU-hours: 3313.6991666666663


In [42]:
low_only_ids = clean_completed_low_ids - clean_completed_mem_ids
mem_only_ids = clean_completed_mem_ids - clean_completed_low_ids
dual_signal_ids = clean_completed_low_ids & clean_completed_mem_ids

print("Low-util only:", len(low_only_ids))
print("Memory-oversized only:", len(mem_only_ids))
print("Both signals:", len(dual_signal_ids))
print("Total unique:", len(
    clean_completed_low_ids | clean_completed_mem_ids
))

Low-util only: 6
Memory-oversized only: 6
Both signals: 8
Total unique: 20


In [43]:
def summarize_jobs(ids, label):
    df = jobs[jobs["id_job"].isin(ids)]
    return {
        "group": label,
        "jobs": len(ids),
        "gpu_hours": df["gpu_hours"].sum(),
        "usd_full_spend": df["gpu_hours"].sum() * 2.5
    }

rightsizing_summary = pd.DataFrame([
    summarize_jobs(low_only_ids, "Low-util only"),
    summarize_jobs(mem_only_ids, "Memory oversized only"),
    summarize_jobs(dual_signal_ids, "Both signals")
])

rightsizing_summary

,group,jobs,gpu_hours,usd_full_spend
0,Low-util only,6,3492.683111,8731.707778
1,Memory oversized only,6,2153.426944,5383.567361
2,Both signals,8,3313.699167,8284.247917


In [44]:
print("Total FAILED jobs:",
      (jobs["state_name"] == "FAILED").sum())

print("Jobs that hit node failure:",
      jobs["hit_node_failure"].sum())

print(
    jobs.groupby("state_name")["hit_node_failure"]
        .agg(["count", "sum"])
)

Total FAILED jobs: 18587
Jobs that hit node failure: 31
                count  sum
state_name                
CANCELLED        9290    7
COMPLETED       45334    6
FAILED          18587    5
NODE_FAIL          10   10
TIMEOUT          1544    3
UNDECODED_1024      1    0
UNDECODED_11       83    0


In [45]:
node_failure_jobs = jobs[
    jobs["hit_node_failure"] == True
]

print("Jobs that hit node failure:", len(node_failure_jobs))
print("GPU-hours:", node_failure_jobs["gpu_hours"].sum())
print("USD:", node_failure_jobs["gpu_hours"].sum() * 2.5)

display(
    node_failure_jobs[
        [
            "id_job",
            "state_name",
            "gpu_hours",
            "primary_node",
            "nodefail_nodes"
        ]
    ].head()
)

Jobs that hit node failure: 31
GPU-hours: 8018.693413888889
USD: 20046.73353472222


,id_job,state_name,gpu_hours,primary_node,nodefail_nodes
1533,1369901951,COMPLETED,14.106389,r7753495-n772143,[r4683026-n772143]
4357,3786120982,TIMEOUT,24.006167,r2684277-n303509,[r7419443-n303509]
13513,12042739192,COMPLETED,4.861806,r974863-n410412,[r216287-n172107]
16334,14550591462,TIMEOUT,24.013556,r1900814-n172107,[r1039410-n303509]
18838,16800406571,CANCELLED,35.629611,r8024255-n410412,[r810901-n772143]


In [46]:
hardware_fault_findings = [
    f for f in findings
    if f.get("detectorId") == "rules::node-hardware-fault"
]

print("Hardware fault findings:", len(hardware_fault_findings))

Hardware fault findings: 1


In [47]:
hardware_fault_hours = sum(
    f.get("metadata", {}).get("impact_gpu_hours", 0) or 0
    for f in hardware_fault_findings
)

print("Silent hardware fault GPU-hours:", hardware_fault_hours)
print("USD:", hardware_fault_hours * 2.5)

Silent hardware fault GPU-hours: 76.62
USD: 191.55


In [48]:
node_failure_findings = [
    f for f in findings
    if f.get("detectorId") == "rules::node-failure"
]

node_failure_impact_hours = sum(
    f.get("metadata", {}).get("impact_gpu_hours", 0) or 0
    for f in node_failure_findings
)

print("Node-failure findings:", len(node_failure_findings))
print("Scheduler-recorded lost GPU-hours:", node_failure_impact_hours)
print("USD:", node_failure_impact_hours * 2.5)
print(
    "% total capacity:",
    node_failure_impact_hours / total_gpu_hours * 100
)

Node-failure findings: 31
Scheduler-recorded lost GPU-hours: 7772.07
USD: 19430.175
% total capacity: 1.3084208341330013


In [49]:
hardware_fault_findings[0]

{'id': 'e0c7d180-640e-5f03-8bfc-e51cca3df21e',
 'integrationId': 'dc59fe21-aef3-56fb-a6e5-7de22e87047d',
 'enterpriseId': '17372250-e094-56f6-9f3c-3907f9017fd1',
 'detectorId': 'rules::node-hardware-fault',
 'shortDescription': 'r216287-n200569: SIGBUS (bus error) crashes that follow the machine, not the people',
 'longDescription': 'Between 2026-02-27T16:00:36+00:00 and 2026-03-07T16:00:36+00:00, 3 different people had jobs crash on r216287-n200569 with exit status 135 (SIGBUS (bus error)). None of them produces that status anywhere else: u-16337070303: 23 here, 0 in 1,427 jobs on every other machine; u-32801634342: 5 here, 0 in 1,544 jobs on every other machine; u-56073333661: 86 here, 0 in 946 jobs on every other machine. A failure that appears for several unrelated people on one machine and for none of them elsewhere is a property of the machine. The scheduler never marked it down, so it kept receiving work throughout -- 140 of 144 jobs failed.',
 'impactDescription': '76.6 GPU-hou